In [93]:
import zipfile

ZIP_PATH = "/content/archive.zip"

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    csv_files = [f for f in z.namelist() if f.endswith(".csv")]

# Exclude non-stock files
exclude_files = ["NIFTY50_all.csv", "stock_metadata.csv"]
csv_files = [f for f in csv_files if f.split("/")[-1] not in exclude_files]

print("Total CSV files:", len(csv_files))
csv_files[:10]


Total CSV files: 50


['ADANIPORTS.csv',
 'ASIANPAINT.csv',
 'AXISBANK.csv',
 'BAJAJ-AUTO.csv',
 'BAJAJFINSV.csv',
 'BAJFINANCE.csv',
 'BHARTIARTL.csv',
 'BPCL.csv',
 'BRITANNIA.csv',
 'CIPLA.csv']

In [94]:
import pandas as pd

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    with z.open(csv_files[0]) as f:
        df_test = pd.read_csv(f)

df_test.head()

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2007-11-27,MUNDRAPORT,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,27294366,2.687719e+15,NaN,9859619,0.3612
1,2007-11-28,MUNDRAPORT,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,4581338,4.312765e+14,NaN,1453278,0.3172
2,2007-11-29,MUNDRAPORT,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,5124121,4.550658e+14,NaN,1069678,0.2088
3,2007-11-30,MUNDRAPORT,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,4609762,4.283257e+14,NaN,1260913,0.2735
4,2007-12-03,MUNDRAPORT,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,2977470,2.875200e+14,NaN,816123,0.2741


In [95]:
import numpy as np

summary = []

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for file in csv_files:
        try:
            with z.open(file) as f:
                df = pd.read_csv(f)

            if not {"Date", "Close", "Volume"}.issubset(df.columns):
                continue

            df["Date"] = pd.to_datetime(df["Date"])
            df.sort_values("Date", inplace=True)
            df.dropna(subset=["Close", "Volume"], inplace=True)

            summary.append({
                "Stock": file.replace(".csv", "").split("/")[-1],
                "Avg_Close": df["Close"].mean(),
                "Total_Volume": df["Volume"].sum(),
                "Volatility": df["Close"].pct_change().std(),
                "Data_Points": len(df)
            })

        except Exception as e:
            print(f"Skipped {file}: {e}")

summary_df = pd.DataFrame(summary)
summary_df.head()

,Stock,Avg_Close,Total_Volume,Volatility,Data_Points
0,ADANIPORTS,344.201626,9815061348,0.030370,3322
1,ASIANPAINT,1247.410903,2704320093,0.022355,5306
2,AXISBANK,585.893931,24025237150,0.030541,5306
3,BAJAJ-AUTO,2190.412196,1317507462,0.021345,3202
4,BAJAJFINSV,2758.657451,741131260,0.025257,3201


In [96]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
metrics = ["Avg_Close", "Total_Volume", "Volatility", "Data_Points"]

summary_scaled = summary_df.copy()
summary_scaled[metrics] = scaler.fit_transform(summary_df[metrics])

summary_scaled["Score"] = (
    0.4 * summary_scaled["Total_Volume"] +
    0.3 * summary_scaled["Data_Points"] +
    0.2 * summary_scaled["Avg_Close"] +
    0.1 * summary_scaled["Volatility"]
)

In [97]:
top_10_stocks = summary_scaled.sort_values("Score", ascending=False).head(10)
top_10_stocks[["Stock", "Score"]]

,Stock,Score
40,TATAMOTORS,0.782642
37,SBIN,0.762444
21,ICICIBANK,0.697098
47,VEDL,0.637189
26,ITC,0.627411
19,HINDALCO,0.617430
41,TATASTEEL,0.603972
36,RELIANCE,0.579691
49,ZEEL,0.574956
2,AXISBANK,0.559495


In [98]:
top_10_stocks.to_csv("top_10_stocks_selected.csv", index=False)

In [99]:
import pandas as pd

top_10_df = pd.read_csv("top_10_stocks_selected.csv")
top_10_stocks = top_10_df["Stock"].tolist()

top_10_stocks

['TATAMOTORS',
 'SBIN',
 'ICICIBANK',
 'VEDL',
 'ITC',
 'HINDALCO',
 'TATASTEEL',
 'RELIANCE',
 'ZEEL',
 'AXISBANK']

In [100]:
import zipfile

ZIP_PATH = "/content/archive.zip"

def load_stock_from_zip(stock_name):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        with z.open(f"{stock_name}.csv") as f:
            df = pd.read_csv(f)

    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)
    df = df[["Date", "Close", "Volume"]]
    df.dropna(inplace=True)
    return df

In [101]:
eda_summary = []

for stock in top_10_stocks:
    df = load_stock_from_zip(stock)

    eda_summary.append({
        "Stock": stock,
        "Start_Date": df["Date"].min(),
        "End_Date": df["Date"].max(),
        "Avg_Close": df["Close"].mean(),
        "Volatility": df["Close"].pct_change().std(),
        "Total_Volume": df["Volume"].sum()
    })
eda_df = pd.DataFrame(eda_summary)
eda_df

,Stock,Start_Date,End_Date,Avg_Close,Volatility,Total_Volume
0,TATAMOTORS,2000-01-03,2021-04-30,409.450264,0.030746,55530448331
1,SBIN,2000-01-03,2021-04-30,965.895543,0.026841,53268535759
2,ICICIBANK,2000-01-03,2021-04-30,550.995524,0.029979,43639890550
3,VEDL,2000-01-03,2021-04-30,462.489378,0.035807,31726226822
4,ITC,2000-01-03,2021-04-30,420.273690,0.024646,38060813600
5,HINDALCO,2000-01-03,2021-04-30,355.653873,0.029652,33536569673
6,TATASTEEL,2000-01-03,2021-04-30,403.553703,0.027839,32712834078
7,RELIANCE,2000-01-03,2021-04-30,1011.316839,0.024034,29623544140
8,ZEEL,2000-01-03,2021-04-30,273.233566,0.033125,25603688489
9,AXISBANK,2000-01-03,2021-04-30,585.893931,0.030541,24025237150


In [102]:
import numpy as np
import pandas as pd
import zipfile
import plotly.graph_objects as go

def load_stock_from_zip(stock_name):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        with z.open(f"{stock_name}.csv") as f:
            df = pd.read_csv(f)

    df["Date"] = pd.to_datetime(df["Date"])
    df.sort_values("Date", inplace=True)
    df = df[["Date", "Close", "Volume"]]
    df.dropna(inplace=True)
    return df


In [103]:
def plot_price_trend(stock):
    df = load_stock_from_zip(stock)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["Date"],
        y=df["Close"],
        name="Close Price"
    ))

    fig.update_layout(
        title=f"{stock} – Closing Price Trend",
        xaxis_title="Date",
        yaxis_title="Price"
    )

    fig.show()

In [104]:
def plot_daily_returns(stock):
    df = load_stock_from_zip(stock)
    df["Returns"] = df["Close"].pct_change()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["Date"],
        y=df["Returns"],
        name="Daily Returns"
    ))

    fig.update_layout(
        title=f"{stock} – Daily Returns",
        xaxis_title="Date",
        yaxis_title="Returns"
    )

    fig.show()

In [105]:
def plot_rolling_volatility(stock, window=30):
    df = load_stock_from_zip(stock)
    df["Returns"] = df["Close"].pct_change()
    df["Rolling_Volatility"] = df["Returns"].rolling(window).std()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["Date"],
        y=df["Rolling_Volatility"],
        name="Rolling Volatility"
    ))

    fig.update_layout(
        title=f"{stock} – {window}-Day Rolling Volatility",
        xaxis_title="Date",
        yaxis_title="Volatility"
    )

    fig.show()

In [106]:
def train_test_split_ts(df, test_size=0.2):
    split_idx = int(len(df) * (1 - test_size))
    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]
    return train, test

In [107]:
stock = top_10_stocks[0]
df = load_stock_from_zip(stock)

train_df, test_df = train_test_split_ts(df)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

train_df.tail(), test_df.head()

Train size: (4244, 3)
Test size: (1062, 3)


(           Date   Close   Volume
 4239 2017-01-06  497.75  4907475
 4240 2017-01-09  500.15  3932710
 4241 2017-01-10  516.25  7973024
 4242 2017-01-11  519.25  4886398
 4243 2017-01-12  518.25  4219627,
            Date   Close   Volume
 4244 2017-01-13  514.85  6367248
 4245 2017-01-16  526.40  6084582
 4246 2017-01-17  523.70  4297426
 4247 2017-01-18  522.45  5874608
 4248 2017-01-19  531.45  5810128)

In [108]:
!pip install pmdarima

In [109]:
from pmdarima import auto_arima
import numpy as np

def run_auto_arima(train_df, steps):
    model = auto_arima(
        train_df["Close"],
        seasonal=False,
        trace=False,
        suppress_warnings=True
    )
    forecast = model.predict(n_periods=steps)
    return forecast

In [110]:
steps = len(test_df)
arima_preds = run_auto_arima(train_df, steps)
arima_preds[:5]

,0
4244,517.334364
4245,516.722898
4246,516.429795
4247,516.433331
4248,516.693901


In [111]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(test_df["Close"], arima_preds))
print(f"Auto-ARIMA RMSE for {stock}: {rmse:.2f}")

Auto-ARIMA RMSE for TATAMOTORS: 293.03


In [112]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def create_lag_features(df, lags=10):
    df_lag = df.copy()
    for i in range(1, lags + 1):
        df_lag[f"lag_{i}"] = df_lag["Close"].shift(i)
    df_lag.dropna(inplace=True)
    return df_lag

In [113]:
# Using the same stock and split from previous step
df = load_stock_from_zip(stock)
train_df, test_df = train_test_split_ts(df)

train_lag = create_lag_features(train_df)
test_lag = create_lag_features(pd.concat([train_df.tail(10), test_df]))

X_train = train_lag.drop(columns=["Date", "Close", "Volume"])
y_train = train_lag["Close"]

X_test = test_lag.drop(columns=["Date", "Close", "Volume"])
y_test = test_lag["Close"]

In [114]:
rf_model = RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)
rf_model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)

In [115]:
rf_preds = rf_model.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
print(f"Random Forest RMSE for {stock}: {rf_rmse:.2f}")


Random Forest RMSE for TATAMOTORS: 7.19


In [116]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_squared_error

In [117]:
def prepare_lstm_data(series, window=60):
    X, y = [], []
    for i in range(window, len(series)):
        X.append(series[i-window:i])
        y.append(series[i])
    return np.array(X), np.array(y)

In [118]:
# reuse same stock
df = load_stock_from_zip(stock)

scaler = MinMaxScaler()
scaled_close = scaler.fit_transform(df[["Close"]])
split_idx = int(len(scaled_close) * 0.8)

train_series = scaled_close[:split_idx]
test_series = scaled_close[split_idx:]

In [119]:
WINDOW = 60

X_train, y_train = prepare_lstm_data(train_series, WINDOW)
X_test, y_test = prepare_lstm_data(test_series, WINDOW)

# reshape for LSTM [samples, timesteps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

In [120]:
lstm_model = Sequential([LSTM(50, return_sequences=True, input_shape=(WINDOW, 1)),LSTM(50),Dense(1)])
lstm_model.compile(optimizer="adam", loss="mse")

lstm_model.fit(X_train, y_train,epochs=10, batch_size=32,verbose=1)
lstm_preds = lstm_model.predict(X_test)

# inverse scaling
lstm_preds_inv = scaler.inverse_transform(lstm_preds)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

lstm_rmse = np.sqrt(mean_squared_error(y_test_inv, lstm_preds_inv))
print(f"LSTM RMSE for {stock}: {lstm_rmse:.2f}")

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



131/131 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - loss: 0.0146
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 7.8964e-04
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 7.5608e-04
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - loss: 5.3631e-04
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 11s 54ms/step - loss: 4.7206e-04
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 6.2120e-04
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - loss: 3.7713e-04
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 55ms/step - loss: 5.5901e-04
Epoch 9/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 10s 55ms/step - loss: 3.8114e-04
Epoch 10/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - loss: 4.4015e-04
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step
LSTM RMSE for TATAMOTORS: 11.83


In [121]:
comparison_df = pd.DataFrame({
    "Stock": [stock],
    "Auto_ARIMA_RMSE": [rmse],
    "Random_Forest_RMSE": [rf_rmse],
    "LSTM_RMSE": [lstm_rmse]
})

comparison_df

,Stock,Auto_ARIMA_RMSE,Random_Forest_RMSE,LSTM_RMSE
0,TATAMOTORS,293.033178,7.194901,11.830536


In [122]:
def get_best_model(row):
    rmses = {
        "Auto_ARIMA": row["Auto_ARIMA_RMSE"],
        "Random_Forest": row["Random_Forest_RMSE"],
        "LSTM": row["LSTM_RMSE"]
    }
    return min(rmses, key=rmses.get)

comparison_df["Best_Model"] = comparison_df.apply(get_best_model, axis=1)
comparison_df

,Stock,Auto_ARIMA_RMSE,Random_Forest_RMSE,LSTM_RMSE,Best_Model
0,TATAMOTORS,293.033178,7.194901,11.830536,Random_Forest


In [123]:
def forecast_arima_future(df, days=30):
    model = auto_arima(df["Close"], seasonal=False, suppress_warnings=True)
    return model.predict(n_periods=days)

In [124]:
def forecast_rf_future(df, days=30, lags=10):
    df_lag = create_lag_features(df.copy(), lags)

    X = df_lag.drop(columns=["Date", "Close", "Volume"])
    y = df_lag["Close"]

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X, y)

    # keep column names
    last_row = X.iloc[-1:].copy()
    preds = []

    for _ in range(days):
        pred = model.predict(last_row)[0]
        preds.append(pred)

        # shift lag values
        new_row = last_row.shift(-1, axis=1)
        new_row.iloc[0, -1] = pred
        last_row = new_row

    return preds

In [125]:
def forecast_lstm_future(df, days=30, window=60):
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[["Close"]])

    X, y = prepare_lstm_data(scaled, window)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    model = Sequential([
        Input(shape=(window, 1)),
        LSTM(50, return_sequences=True),
        LSTM(50),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    model.fit(X, y, epochs=10, batch_size=32, verbose=0)

    last_seq = scaled[-window:]
    future = []

    for _ in range(days):
        pred = model.predict(last_seq.reshape(1, window, 1), verbose=0)
        future.append(pred[0, 0])
        last_seq = np.vstack([last_seq[1:], pred])

    return scaler.inverse_transform(
        np.array(future).reshape(-1, 1)
    ).flatten()

In [126]:
df = load_stock_from_zip(stock)

best_model = comparison_df.loc[
    comparison_df["Stock"] == stock, "Best_Model"
].values[0]

if best_model == "Auto_ARIMA":
    future_preds = forecast_arima_future(df)
elif best_model == "Random_Forest":
    future_preds = forecast_rf_future(df)
else:
    future_preds = forecast_lstm_future(df)

future_preds[:5]

[np.float64(297.8215000000007),
 np.float64(302.53333333333296),
 np.float64(302.7294999999999),
 np.float64(296.7519999999999),
 np.float64(293.7548333333331)]

In [127]:
import pandas as pd

def get_future_dates(df, days=30):
    last_date = df["Date"].iloc[-1]
    future_dates = pd.date_range(
        start=last_date,
        periods=days + 1,
        freq="B"  # business days
    )[1:]
    return future_dates

In [128]:
import plotly.graph_objects as go

def plot_future_forecast(df, future_preds, stock, model_name):
    future_dates = get_future_dates(df, len(future_preds))

    fig = go.Figure()

    # Historical prices
    fig.add_trace(go.Scatter(
        x=df["Date"],
        y=df["Close"],
        name="Historical Close",
        line=dict(color="blue")
    ))

    # Future predictions
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=future_preds,
        name=f"Future Forecast ({model_name})",
        line=dict(color="red", dash="dash")
    ))

    fig.update_layout(
        title=f"{stock} – 30 Day Stock Price Forecast ({model_name})",
        xaxis_title="Date",
        yaxis_title="Price",
        legend_title="Legend"
    )

    fig.show()

In [129]:
import plotly.graph_objects as go

def save_forecast_plot(df, future_preds, stock, model_name):
    future_dates = get_future_dates(df, len(future_preds))

    fig = go.Figure()

    # Historical prices
    fig.add_trace(go.Scatter(
        x=df["Date"],
        y=df["Close"],
        name="Historical Close"
    ))

    # Future forecast
    fig.add_trace(go.Scatter(
        x=future_dates,
        y=future_preds,
        name=f"Forecast ({model_name})",
        line=dict(dash="dash")
    ))

    fig.update_layout(
        title=f"{stock} – 30 Day Forecast ({model_name})",
        xaxis_title="Date",
        yaxis_title="Price"
    )

    file_path = f"outputs/plots_html/{stock}_forecast.html"
    fig.write_html(file_path)

    return file_path

In [130]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
comparison_rows = []

for stock in top_10_stocks:
    print(f"Training models for: {stock}")

    df = load_stock_from_zip(stock)
    train_df, test_df = train_test_split_ts(df)

    # -------- Auto-ARIMA --------
    arima_preds = run_auto_arima(train_df, len(test_df))
    arima_rmse = np.sqrt(mean_squared_error(test_df["Close"], arima_preds))

    # -------- Random Forest --------
    train_lag = create_lag_features(train_df)
    test_lag = create_lag_features(pd.concat([train_df.tail(10), test_df]))

    X_train = train_lag.drop(columns=["Date", "Close", "Volume"])
    y_train = train_lag["Close"]
    X_test = test_lag.drop(columns=["Date", "Close", "Volume"])
    y_test = test_lag["Close"]

    rf = RandomForestRegressor(n_estimators=300, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)
    rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

    # -------- LSTM --------
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df[["Close"]])

    split = int(len(scaled) * 0.8)
    train_s, test_s = scaled[:split], scaled[split:]

    Xtr, ytr = prepare_lstm_data(train_s, 60)
    Xte, yte = prepare_lstm_data(test_s, 60)

    Xtr = Xtr.reshape(Xtr.shape[0], Xtr.shape[1], 1)
    Xte = Xte.reshape(Xte.shape[0], Xte.shape[1], 1)

    lstm = Sequential([
    Input(shape=(60, 1)),
    LSTM(50, return_sequences=True),
    LSTM(50),
    Dense(1)])


    lstm.compile(optimizer="adam", loss="mse")
    lstm.fit(Xtr, ytr, epochs=5, batch_size=32, verbose=0)

    lstm_preds = lstm.predict(Xte, verbose=0)
    lstm_rmse = np.sqrt(
        mean_squared_error(
            scaler.inverse_transform(yte.reshape(-1, 1)),
            scaler.inverse_transform(lstm_preds)
        )
    )

    comparison_rows.append({
        "Stock": stock,
        "Auto_ARIMA_RMSE": arima_rmse,
        "Random_Forest_RMSE": rf_rmse,
        "LSTM_RMSE": lstm_rmse
    })

Training models for: TATAMOTORS
Training models for: SBIN
Training models for: ICICIBANK
Training models for: VEDL
Training models for: ITC
Training models for: HINDALCO
Training models for: TATASTEEL
Training models for: RELIANCE
Training models for: ZEEL
Training models for: AXISBANK


In [131]:
comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Stock,Auto_ARIMA_RMSE,Random_Forest_RMSE,LSTM_RMSE
0,TATAMOTORS,293.033178,7.194901,13.456876
1,SBIN,57.815444,8.226985,17.553463
2,ICICIBANK,145.153296,10.557895,17.499191
3,VEDL,80.716325,5.677156,22.638333
4,ITC,42.005525,4.827909,31.072608
5,HINDALCO,57.541501,20.904405,11.846802
6,TATASTEEL,153.239155,14.701008,22.226830
7,RELIANCE,489.568344,43.004049,80.180072
8,ZEEL,1561.846538,13.469179,16.472265
9,AXISBANK,175.609109,15.185949,32.585145


In [132]:
def select_best_model(row):
    rmses = {
        "Auto_ARIMA": row["Auto_ARIMA_RMSE"],
        "Random_Forest": row["Random_Forest_RMSE"],
        "LSTM": row["LSTM_RMSE"]
    }
    return min(rmses, key=rmses.get)

comparison_df["Best_Model"] = comparison_df.apply(select_best_model, axis=1)
comparison_df

,Stock,Auto_ARIMA_RMSE,Random_Forest_RMSE,LSTM_RMSE,Best_Model
0,TATAMOTORS,293.033178,7.194901,13.456876,Random_Forest
1,SBIN,57.815444,8.226985,17.553463,Random_Forest
2,ICICIBANK,145.153296,10.557895,17.499191,Random_Forest
3,VEDL,80.716325,5.677156,22.638333,Random_Forest
4,ITC,42.005525,4.827909,31.072608,Random_Forest
5,HINDALCO,57.541501,20.904405,11.846802,LSTM
6,TATASTEEL,153.239155,14.701008,22.226830,Random_Forest
7,RELIANCE,489.568344,43.004049,80.180072,Random_Forest
8,ZEEL,1561.846538,13.469179,16.472265,Random_Forest
9,AXISBANK,175.609109,15.185949,32.585145,Random_Forest


In [133]:
import os
import pandas as pd

os.makedirs("outputs/forecasts", exist_ok=True)
os.makedirs("outputs/plots_html", exist_ok=True)

final_results = []

for _, row in comparison_df.iterrows():
    stock = row["Stock"]
    best_model = row["Best_Model"]

    df = load_stock_from_zip(stock)

    if best_model == "Auto_ARIMA":
        future_preds = forecast_arima_future(df)
    elif best_model == "Random_Forest":
        future_preds = forecast_rf_future(df)
    else:
        future_preds = forecast_lstm_future(df)

    # Save plot
    plot_path = save_forecast_plot(df, future_preds, stock, best_model)

    # Save forecast CSV
    forecast_df = pd.DataFrame({
        "Date": get_future_dates(df, len(future_preds)),
        "Predicted_Close": future_preds
    })

    forecast_csv = f"outputs/forecasts/{stock}_future_30_days.csv"
    forecast_df.to_csv(forecast_csv, index=False)

    final_results.append({
        "Stock": stock,
        "Best_Model": best_model,
        "Forecast_File": forecast_csv,
        "Plot_File": plot_path
    })

In [134]:
final_results_df = pd.DataFrame(final_results)
final_results_df.to_csv("outputs/final_model_results.csv", index=False)

final_results_df

,Stock,Best_Model,Forecast_File,Plot_File
0,TATAMOTORS,Random_Forest,outputs/forecasts/TATAMOTORS_future_30_days.csv,outputs/plots_html/TATAMOTORS_forecast.html
1,SBIN,Random_Forest,outputs/forecasts/SBIN_future_30_days.csv,outputs/plots_html/SBIN_forecast.html
2,ICICIBANK,Random_Forest,outputs/forecasts/ICICIBANK_future_30_days.csv,outputs/plots_html/ICICIBANK_forecast.html
3,VEDL,Random_Forest,outputs/forecasts/VEDL_future_30_days.csv,outputs/plots_html/VEDL_forecast.html
4,ITC,Random_Forest,outputs/forecasts/ITC_future_30_days.csv,outputs/plots_html/ITC_forecast.html
5,HINDALCO,LSTM,outputs/forecasts/HINDALCO_future_30_days.csv,outputs/plots_html/HINDALCO_forecast.html
6,TATASTEEL,Random_Forest,outputs/forecasts/TATASTEEL_future_30_days.csv,outputs/plots_html/TATASTEEL_forecast.html
7,RELIANCE,Random_Forest,outputs/forecasts/RELIANCE_future_30_days.csv,outputs/plots_html/RELIANCE_forecast.html
8,ZEEL,Random_Forest,outputs/forecasts/ZEEL_future_30_days.csv,outputs/plots_html/ZEEL_forecast.html
9,AXISBANK,Random_Forest,outputs/forecasts/AXISBANK_future_30_days.csv,outputs/plots_html/AXISBANK_forecast.html
